# GMI Cloud: Production Grade AI Inference

## Introduction

[GMI Cloud](https://www.gmicloud.ai/?utm_source=aibuilders) is a production-grade AI inference platform that delivers enterprise-scale model serving with guaranteed performance and flexibility. Whether you're prototyping a new AI feature or running mission-critical workloads at scale, GMI Cloud provides the infrastructure to deploy and run large language models efficiently.

![GMI Cloud](/products/gmicloud/gmicloud_1.jpg)



### Deployment Options: Serverless vs. Dedicated

GMI Cloud offers two deployment models:

#### Serverless (Pay-As-You-Go)

Pay per token (e.g., $1.00/1M input, $3.20/1M output). Auto-scales to zero when idle.

- **Pros:** No setup, no idle costs, instant access
- **Cons:** Possible cold starts, shared infrastructure
- **Best for:** Prototyping, variable traffic, low-to-medium volume

#### Dedicated Instance

Rent exclusive GPU hardware (e.g., 8× NVIDIA H200) at a flat hourly rate.

- **Pros:** Guaranteed low latency, no rate limits, full control, enhanced privacy
- **Cons:** Pay for idle time
- **Best for:** Production workloads, high volume, strict security requirements

**Typical journey:** Start **Serverless** for prototyping → migrate to **Dedicated** at scale.

## Prerequisites

Create a `.env` file in the same directory as this notebook with your GMI Cloud API key:

```
GMI_API_KEY=your_gmi_cloud_api_key_here
```

To obtain an API key, sign up at [console.gmicloud.ai](https://console.gmicloud.ai), then navigate to **User / Organization Settings → API Keys** and generate a new key.

In [1]:
# Install required packages
!pip install -q openai python-dotenv requests

In [9]:
import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

GMI_API_KEY = os.getenv("GMI_API_KEY")
if not GMI_API_KEY:
    raise ValueError("GMI_API_KEY not found in .env file")

# GMI Cloud Inference Engine — OpenAI-compatible base URL
BASE_URL = "https://api.gmi-serving.com/v1"
MODEL_NAME = "zai-org/GLM-5-FP8"

# Initialize the OpenAI client pointed at GMI Cloud
client = OpenAI(
    api_key=GMI_API_KEY,
    base_url=BASE_URL,
)

print(f"Client configured — base_url: {BASE_URL}")
print(f"Model: {MODEL_NAME}")

Client configured — base_url: https://api.gmi-serving.com/v1
Model: zai-org/GLM-5-FP8


## 1 — Basic Chat Completion

The simplest use case is sending a single user message and receiving a complete response. This uses the `POST /v1/chat/completions` endpoint, which accepts `model`, `messages`, `temperature`, and `max_tokens` among other parameters.

In [14]:
response = client.chat.completions.create(
    extra_body={"thinking": {"type": "disabled"}},  # Disable thinking mode
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Explain what a Mixture-of-Experts model is in three sentences."},
    ],
    temperature=0.7,
    max_tokens=1000,
)

answer = response.choices[0].message.content
print("GLM-5 says:\n")
print(answer)

# Inspect token usage
usage = response.usage
print(f"\n--- Token usage: prompt={usage.prompt_tokens}, "
      f"completion={usage.completion_tokens}, total={usage.total_tokens}")

GLM-5 says:

A Mixture-of-Experts model is a machine learning architecture that consists of multiple specialized sub-networks, known as experts, which handle different types of input data. A gating network acts as a router to dynamically select which specific experts should be activated for a given input, allowing the model to use only a fraction of its total parameters at a time. This sparse activation enables the model to scale to a massive size and knowledge capacity while maintaining computational efficiency during training and inference.

--- Token usage: prompt=28, completion=609, total=637


## 2 — Structured JSON Output

GLM-5 can return structured data in JSON format, which is ideal for programmatic consumption. Pass `response_format={"type": "json_object"}` and instruct the model to respond in JSON within the prompt.

In [23]:
response = client.chat.completions.create(
    extra_body={"thinking": {"type": "disabled"}},  # Disable thinking mode
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a data extraction assistant. "
                "Always respond with valid JSON."
            ),
        },
        {
            "role": "user",
            "content": (
                "Extract structured information from this text:\n\n"
                "'Apple Inc. reported Q3 2025 revenue of $94.8 billion, "
                "up 5% year-over-year. CEO Tim Cook highlighted strong "
                "growth in the Services segment, which reached $25.2 billion.'"
                "\n\nReturn JSON with keys: company, quarter, year, "
                "revenue_billions, yoy_growth_pct, ceo, highlight_segment, "
                "segment_revenue_billions."
            ),
        },
    ],
    temperature=0,
    max_tokens=1000,
    response_format={"type": "json_object"},
)

# GLM-5 returns content after thinking
raw = response.choices[0].message.content
if not raw:
    # Fallback: content might be empty if model is still in thinking mode
    print("Note: Content was empty, checking raw response...")
    print(response.choices[0].message)
else:
    parsed = json.loads(raw)
    print(json.dumps(parsed, indent=2))

{
  "company": "Apple Inc.",
  "quarter": "Q3",
  "year": 2025,
  "revenue_billions": 94.8,
  "yoy_growth_pct": 5,
  "ceo": "Tim Cook",
  "highlight_segment": "Services",
  "segment_revenue_billions": 25.2
}


## 3 — Code Generation

GLM-5 scores 77.8 on SWE-bench Verified and 56.2 on Terminal Bench 2.0, placing it among the strongest open-weight models for software engineering tasks. Here we ask it to generate a complete, working Python function with docstring and tests.

In [24]:
code_prompt = """
Write a Python function `merge_sorted_lists(a, b)` that merges two sorted
lists into a single sorted list without using the built-in `sorted()` or
`.sort()`. Include:
1. A clear docstring
2. Two assert-based test cases at the bottom
"""

response = client.chat.completions.create(
    extra_body={"thinking": {"type": "disabled"}},  # Disable thinking mode
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": "You are an expert Python developer. Return only code, no extra commentary.",
        },
        {"role": "user", "content": code_prompt},
    ],
    temperature=0,
    max_tokens=2000,
)

generated_code = response.choices[0].message.content
if generated_code:
    print(generated_code)
else:
    print("Note: content is empty, printing raw message...")
    print(response.choices[0].message)

```python
def merge_sorted_lists(a, b):
    """
    Merge two sorted lists into a single sorted list.

    This function uses a two-pointer approach to efficiently merge two
    pre-sorted lists in O(n + m) time complexity, where n and m are the
    lengths of the input lists.

    Args:
        a: First sorted list (ascending order).
        b: Second sorted list (ascending order).

    Returns:
        A new sorted list containing all elements from both input lists
        in ascending order.

    Examples:
        >>> merge_sorted_lists([1, 3, 5], [2, 4, 6])
        [1, 2, 3, 4, 5, 6]
        >>> merge_sorted_lists([1, 2], [3, 4, 5])
        [1, 2, 3, 4, 5]
    """
    result = []
    i, j =


## 4 — Practical Example: Automated Report Summarizer

This final example combines several techniques — system prompts, structured output, and streaming — into a practical report-summarization pipeline. Given a block of text, GLM-5 produces a structured JSON summary, then generates a human-readable executive brief streamed in real time.

In [35]:
report_text = """
Global AI Infrastructure Report — Q4 2025

Cloud GPU demand surged 42% quarter-over-quarter, driven primarily by the
training and inference requirements of large language models exceeding 100B
parameters. NVIDIA H200 GPUs accounted for 58% of new deployments, while
the H100 remained the workhorse for cost-sensitive workloads at $2.00/hr.

Serverless inference adoption grew 67%, with organizations citing automatic
scaling-to-zero and pay-per-token billing as key drivers. Average inference
latency for 70B-parameter models dropped to 38ms per token, a 23%
improvement over Q3.

The open-source model ecosystem saw significant releases: GLM-5 (744B MoE,
40B activated) achieved state-of-the-art scores on SWE-bench and
Terminal-Bench, while DeepSeek-R1 continued to dominate reasoning benchmarks.
Enterprise adoption of open-weight models rose to 41%, up from 29% in Q3.
"""

# Step 1: Structured extraction
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "Extract key metrics from the report. Return JSON with keys: "
                "gpu_demand_growth_pct, top_gpu_model, top_gpu_share_pct, "
                "serverless_growth_pct, avg_latency_ms, open_source_adoption_pct, "
                "notable_models (list of strings)."
            ),
        },
        {"role": "user", "content": report_text},
    ],
    temperature=0,
    max_tokens=1000,
    response_format={"type": "json_object"},
)

# Handle GLM-5's thinking mode
message = response.choices[0].message
raw_content = message.content

if raw_content:
    metrics = json.loads(raw_content)
    print("Extracted Metrics:")
    print(json.dumps(metrics, indent=2))
else:
    # Fallback: try to extract from model_dump
    data = message.model_dump()
    reasoning = data.get('reasoning_content', '')
    print("Note: GLM-5 returned thinking output. Summary of report analysis:")
    # Print last few lines which usually contain the result
    lines = reasoning.strip().split('\n')[-10:]
    print('\n'.join(lines))


Extracted Metrics:
{
  "gpu_demand_growth_pct": 42,
  "top_gpu_model": "NVIDIA H200",
  "top_gpu_share_pct": 58,
  "serverless_growth_pct": 67,
  "avg_latency_ms": 38,
  "open_source_adoption_pct": 41,
  "notable_models": [
    "GLM-5",
    "DeepSeek-R1"
  ]
}


## Summary & Next Steps

In this tutorial you used **GLM-5 via GMI Cloud's Inference Engine** for four common inference patterns: basic chat completion, structured JSON output, code generation, and automated report summarization. Because GMI Cloud exposes an OpenAI-compatible API, migrating existing OpenAI-based code requires only changing the `base_url` and API key.

**Where to go from here:**

- Explore **dedicated endpoints** on GMI Cloud for production workloads with guaranteed GPU capacity and lower latency.
- Scale from **serverless** (pay-per-token) to **dedicated instances** as your token volume grows.
- Use GMI Cloud's **video and image generation APIs** (Veo3, Flux, Seedream) for multimodal pipelines.
- Read the full [GMI Cloud documentation](https://docs.gmicloud.ai) for advanced features.

Happy building!
